In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="fine-tuning",
    notebook_name="05_instruction_tuning.ipynb"
)

# Instruction Tuning: How ChatGPT Learned to Be Helpful

This notebook answers one of the most fascinating questions in AI: **How did we go from a word-predicting model to a helpful assistant like ChatGPT?**

The answer: **Instruction Tuning** + **RLHF** (Reinforcement Learning from Human Feedback).

> **Theory first:** If you have not read [instruction-tuning.md](./instruction-tuning.md), start there. It explains the three-step process (SFT, RLHF, DPO) in plain words with no math. This notebook shows you the ideas in code.

## What You Will Learn
- Why raw language models are not helpful (and what changes that)
- The 3-step recipe that created ChatGPT
- What instruction datasets look like
- How RLHF works (teaching AI with human preferences)
- DPO: a simpler alternative to RLHF
- How to create your own instruction dataset

## Prerequisites
- Completed the intro notebook — [01_what_is_fine_tuning.ipynb](./01_what_is_fine_tuning.ipynb)
- Basic Python knowledge

---
## 1. The Problem: Raw Models Are Not Helpful

A pre-trained language model (like GPT-3 before it became ChatGPT) was trained to do one thing: **predict the next word**.

It read the entire internet and learned patterns, but it does not know how to **have a conversation** or **follow instructions**.

### The Student Analogy

```
  Pre-trained model = A student who has read EVERY book in the library
  
  Knows tons of facts
  Can write grammatically
  Understands many topics
  
  But:
  Does not know how to answer questions helpfully
  Might give dangerous advice casually
  Tends to ramble instead of being concise
  Does not know when to say "I do not know"
```

### What Happens When You Ask a Raw Model a Question

```
  You: "What's the capital of France?"
  
  Raw GPT-3 might say:
  "What's the capital of France? What's the capital of Germany?
   What's the capital of Italy? These are common geography
   questions that students often encounter on..."
  
  It just CONTINUES the text! It does not answer the question.
  It thinks you are writing a document and adds more text.
  
  ChatGPT (instruction-tuned) says:
  "The capital of France is Paris."
  
  Clean. Helpful. Directly answers the question.
```

**Instruction tuning is what bridges this gap.**

---
## 2. The ChatGPT Recipe: Three Steps to a Helpful AI

OpenAI published how they created ChatGPT (via InstructGPT) in 2022. It is a 3-step process:

```
  THE CHATGPT RECIPE
  ══════════════════
  
  Step 1: SUPERVISED FINE-TUNING (SFT)
  ┌─────────────────────────────────────────────────────────┐
  │  Train on thousands of (instruction, ideal response)    │
  │  pairs written by human experts.                        │
  │                                                         │
  │  Example:                                               │
  │  Instruction: "Explain gravity to a 5-year-old"         │
  │  Response: "You know how when you throw a ball up,      │
  │  it always comes back down? That is because the Earth   │
  │  is like a big magnet that pulls everything toward it!" │
  └───────────────────────────┬─────────────────────────────┘
                            │
                            v
  Step 2: REWARD MODEL TRAINING
  ┌─────────────────────────────────────────────────────────┐
  │  Train a separate model to predict which responses      │
  │  humans prefer. Show it pairs of responses and let      │
  │  humans pick the better one.                            │
  │                                                         │
  │  "Which response is better?"                            │
  │  A: "Paris is the capital of France" ← Human picks this │
  │  B: "France capital is Paris city"                      │
  └───────────────────────────┬─────────────────────────────┘
                            │
                            v
  Step 3: RLHF (Reinforcement Learning from Human Feedback)
  ┌─────────────────────────────────────────────────────────┐
  │  Use the reward model to further train the LLM.         │
  │  The model learns to generate responses that the        │
  │  reward model rates highly.                             │
  │                                                         │
  │  Model generates → Reward model scores → Model improves │
  └─────────────────────────────────────────────────────────┘
```

In this notebook, we will focus on **Step 1 (Instruction Tuning)** and give an overview of **Steps 2 & 3 (RLHF)**.

---
## 3. Instruction Datasets: What Do They Look Like?

An instruction dataset is a collection of **(instruction, response)** pairs. The model learns to generate good responses for given instructions.

### Common Dataset Formats

There are three main formats used in the industry:

#### Format 1: Simple Instruction-Response
```json
{
  "instruction": "What is photosynthesis?",
  "response": "Photosynthesis is the process by which plants convert sunlight, water, and carbon dioxide into glucose and oxygen."
}
```

#### Format 2: Instruction with Input Context
```json
{
  "instruction": "Summarize the following text in one sentence.",
  "input": "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris. It is named after engineer Gustave Eiffel, whose company designed and built the tower from 1887 to 1889.",
  "response": "The Eiffel Tower is an iron tower in Paris built by Gustave Eiffel's company between 1887 and 1889."
}
```

#### Format 3: Chat/Conversation Format
```json
{
  "messages": [
    {"role": "system", "content": "You are a helpful cooking assistant."},
    {"role": "user", "content": "How do I boil an egg?"},
    {"role": "assistant", "content": "Place eggs in a pot, cover with water, bring to a boil, then cook for 6-12 minutes depending on how done you want them."}
  ]
}
```

In [ ]:
# Let's look at what a real instruction dataset looks like

# This is a sample from the Alpaca instruction dataset
# (one of the most famous instruction tuning datasets)

sample_dataset = [
    {
        "instruction": "Give three tips for staying healthy.",
        "input": "",
        "output": "1. Eat a balanced diet with plenty of fruits and vegetables.\n"
                  "2. Exercise regularly, aiming for at least 30 minutes per day.\n"
                  "3. Get enough sleep, typically 7-9 hours per night."
    },
    {
        "instruction": "Classify the following sentence as positive, negative, or neutral.",
        "input": "The weather today is absolutely beautiful!",
        "output": "Positive"
    },
    {
        "instruction": "Translate the following English text to French.",
        "input": "Hello, how are you?",
        "output": "Bonjour, comment allez-vous?"
    },
    {
        "instruction": "Explain what machine learning is in simple terms.",
        "input": "",
        "output": "Machine learning is a way for computers to learn from examples "
                  "instead of being explicitly programmed. You show the computer many "
                  "examples of something (like photos of cats and dogs), and it "
                  "figures out the patterns to tell them apart on its own."
    },
    {
        "instruction": "Write a haiku about programming.",
        "input": "",
        "output": "Code flows line by line\nBugs hide in the semicolons\nCoffee fuels the fix"
    },
]

print("Sample Instruction Tuning Dataset")
print("=" * 60)

for i, example in enumerate(sample_dataset, 1):
    print(f"\nExample {i}:")
    print(f"  Instruction: {example['instruction']}")
    if example['input']:
        print(f"  Input:       {example['input']}")
    print(f"  Output:      {example['output'][:80]}..." 
          if len(example['output']) > 80 else f"  Output:      {example['output']}")
    print(f"  {'─' * 56}")

print(f"\nTotal examples shown: {len(sample_dataset)}")
print(f"Real datasets have 10,000 - 1,000,000+ examples!")

In [ ]:
# How instruction data is formatted for training

def format_for_training(example):
    """Convert an instruction example into the format a model sees during training.
    
    The model learns to generate everything after 'Response:'
    given everything before it.
    """
    if example['input']:
        prompt = (
            f"Below is an instruction that describes a task, paired with an input "
            f"that provides further context. Write a response.\n\n"
            f"### Instruction:\n{example['instruction']}\n\n"
            f"### Input:\n{example['input']}\n\n"
            f"### Response:\n{example['output']}"
        )
    else:
        prompt = (
            f"Below is an instruction that describes a task. "
            f"Write a response.\n\n"
            f"### Instruction:\n{example['instruction']}\n\n"
            f"### Response:\n{example['output']}"
        )
    return prompt

# Show what the model actually sees during training
print("What the model sees during training:")
print("=" * 60)

for i, example in enumerate(sample_dataset[:3], 1):
    formatted = format_for_training(example)
    print(f"\n--- Example {i} ---")
    print(formatted)
    print()

print("The model learns to generate the Response given the Instruction!")
print("After seeing thousands of these, it becomes an instruction-follower.")

---
## 4. Famous Instruction Datasets

Here are some well-known instruction datasets that have been used to train popular models:

| Dataset | Size | Description | Used By |
|---------|------|-------------|--------|
| **Alpaca** | 52K | Generated by GPT-4, covers diverse tasks | Stanford Alpaca |
| **Dolly** | 15K | Human-written by Databricks employees | Dolly 2.0 |
| **FLAN** | 1.8M+ | Collection of many NLP tasks formatted as instructions | Google FLAN-T5 |
| **ShareGPT** | 90K+ | Real conversations shared by ChatGPT users | Vicuna |
| **OpenAssistant** | 161K | Crowd-sourced conversations with quality ratings | Open-source models |
| **UltraChat** | 1.5M | Multi-turn conversations generated by AI | Zephyr |

### Types of Instructions

A good instruction dataset covers many types of tasks:

```
  Instruction Categories:
  
  ┌────────────────────┐  ┌────────────────────┐  ┌────────────────────┐
  │  Knowledge Q&A     │  │  Creative Writing   │  │  Analysis          │
  │  "What is DNA?"    │  │  "Write a poem"     │  │  "Compare X and Y" │
  └────────────────────┘  └────────────────────┘  └────────────────────┘
  
  ┌────────────────────┐  ┌────────────────────┐  ┌────────────────────┐
  │  Coding            │  │  Math               │  │  Summarization     │
  │  "Write a function"│  │  "Solve 2x + 3 = 7"│  │  "Summarize this"  │
  └────────────────────┘  └────────────────────┘  └────────────────────┘
  
  ┌────────────────────┐  ┌────────────────────┐  ┌────────────────────┐
  │  Translation       │  │  Classification     │  │  Conversation      │
  │  "Translate to..." │  │  "Is this positive?"│  │  Multi-turn chat   │
  └────────────────────┘  └────────────────────┘  └────────────────────┘
```

In [ ]:
# Visualize the instruction tuning pipeline

import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(16, 9))
ax.set_xlim(0, 16)
ax.set_ylim(0, 10)

# -- Stage 1: Pre-training --
box1 = patches.FancyBboxPatch((0.5, 7), 4, 2.2, boxstyle='round,pad=0.3',
                                facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2)
ax.add_patch(box1)
ax.text(2.5, 8.5, 'Stage 1: Pre-Training', ha='center', fontsize=11, fontweight='bold')
ax.text(2.5, 7.9, 'Train on internet text', ha='center', fontsize=9)
ax.text(2.5, 7.4, '(trillions of words)', ha='center', fontsize=8, color='gray')

# -- Stage 2: SFT --
box2 = patches.FancyBboxPatch((6, 7), 4, 2.2, boxstyle='round,pad=0.3',
                                facecolor='#FFF3E0', edgecolor='#E65100', linewidth=2)
ax.add_patch(box2)
ax.text(8, 8.5, 'Stage 2: Instruction Tuning', ha='center', fontsize=11, fontweight='bold')
ax.text(8, 7.9, 'Train on (instruction, response)', ha='center', fontsize=9)
ax.text(8, 7.4, '(10K-1M examples)', ha='center', fontsize=8, color='gray')

# -- Stage 3: RLHF --
box3 = patches.FancyBboxPatch((11.5, 7), 4, 2.2, boxstyle='round,pad=0.3',
                                facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=2)
ax.add_patch(box3)
ax.text(13.5, 8.5, 'Stage 3: RLHF', ha='center', fontsize=11, fontweight='bold')
ax.text(13.5, 7.9, 'Learn from human preferences', ha='center', fontsize=9)
ax.text(13.5, 7.4, '(which answer is better?)', ha='center', fontsize=8, color='gray')

# Arrows between stages
ax.annotate('', xy=(6, 8.1), xytext=(4.5, 8.1),
            arrowprops=dict(arrowstyle='->', color='gray', lw=2))
ax.annotate('', xy=(11.5, 8.1), xytext=(10, 8.1),
            arrowprops=dict(arrowstyle='->', color='gray', lw=2))

# -- What each stage produces --
r1 = patches.FancyBboxPatch((0.5, 4), 4, 2.2, boxstyle='round,pad=0.3',
                              facecolor='#BBDEFB', edgecolor='#1565C0', linewidth=1.5)
ax.add_patch(r1)
ax.text(2.5, 5.5, 'Base Model', ha='center', fontsize=10, fontweight='bold')
ax.text(2.5, 4.9, '"Knows a lot but', ha='center', fontsize=8)
ax.text(2.5, 4.4, 'not helpful"', ha='center', fontsize=8)

r2 = patches.FancyBboxPatch((6, 4), 4, 2.2, boxstyle='round,pad=0.3',
                              facecolor='#FFE0B2', edgecolor='#E65100', linewidth=1.5)
ax.add_patch(r2)
ax.text(8, 5.5, 'SFT Model', ha='center', fontsize=10, fontweight='bold')
ax.text(8, 4.9, '"Follows instructions', ha='center', fontsize=8)
ax.text(8, 4.4, 'but not always well"', ha='center', fontsize=8)

r3 = patches.FancyBboxPatch((11.5, 4), 4, 2.2, boxstyle='round,pad=0.3',
                              facecolor='#C8E6C9', edgecolor='#2E7D32', linewidth=1.5)
ax.add_patch(r3)
ax.text(13.5, 5.5, 'ChatGPT!', ha='center', fontsize=12, fontweight='bold', color='#2E7D32')
ax.text(13.5, 4.9, '"Helpful, harmless,', ha='center', fontsize=8)
ax.text(13.5, 4.4, 'and honest"', ha='center', fontsize=8)

# Arrows from stages to results
for x in [2.5, 8, 13.5]:
    ax.annotate('', xy=(x, 6.2), xytext=(x, 7),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

# Quality indicators
qualities = [
    (2.5, 'Quality: *', '#1565C0'),
    (8, 'Quality: **', '#E65100'),
    (13.5, 'Quality: ***', '#2E7D32')
]
for x, text, color in qualities:
    ax.text(x, 3.5, text, ha='center', fontsize=11, fontweight='bold', color=color)

# Examples at bottom
examples = [
    (2.5, '"Paris France capital\ncity Europe river Seine..."', '#BBDEFB'),
    (8, '"The capital of France\nis Paris."', '#FFE0B2'),
    (13.5, '"Paris is the capital of\nFrance! It\'s known for the\nEiffel Tower and..."', '#C8E6C9')
]
for x, text, color in examples:
    box = patches.FancyBboxPatch((x-2, 0.5), 4, 2.2, boxstyle='round,pad=0.3',
                                  facecolor=color, edgecolor='gray', linewidth=1, alpha=0.5)
    ax.add_patch(box)
    ax.text(x, 1.6, text, ha='center', fontsize=7, fontstyle='italic')

ax.text(8, 0.2, 'User asks: "What is the capital of France?"', ha='center',
        fontsize=10, fontweight='bold', color='gray')

ax.set_title('The 3-Stage Pipeline: From Raw Model to ChatGPT',
             fontsize=16, fontweight='bold', pad=15)
ax.axis('off')

plt.tight_layout()
plt.show()

print("This is the 3-stage process that turned GPT into ChatGPT!")

---
## 5. RLHF: Learning from Human Preferences

**RLHF (Reinforcement Learning from Human Feedback)** is the technique that makes models like ChatGPT feel so natural and helpful.

### How RLHF Works (Step by Step)

```
  Step 1: Collect Human Preferences
  ──────────────────────────────────
  Show humans two responses to the same prompt.
  Ask: "Which is better?"
  
  Prompt: "Explain gravity simply"
  
  Response A: "It is the force that       Response B: "According to general
  pulls things toward each other.          relativity, spacetime curvature
  That is why apples fall from trees       caused by mass-energy..."
  and why you stay on the ground!"
  
  Human: "A is better!"                   (too complicated for 'simply')
  
  
  Step 2: Train a Reward Model
  ──────────────────────────────────
  Using thousands of these preferences, train a model that
  predicts which response a human would prefer.
  
  Reward Model(Response A) = 0.9  (high score = humans like it)
  Reward Model(Response B) = 0.3  (low score = humans do not prefer it)
  
  
  Step 3: Use RL to Optimize the LLM
  ──────────────────────────────────
  Generate responses → Score with reward model → Update LLM
  
  ┌──────────┐    ┌──────────────┐    ┌─────────────┐
  │  LLM     │───>│  Response     │───>│  Reward     │──> Score
  │(generates)│   │  "Gravity.." │    │  Model      │     │
  └──────────┘    └──────────────┘    └─────────────┘     │
       ^                                                   │
       └───── Update weights based on score ───────────────┘
  
  The LLM learns to generate responses that get HIGH scores
  (responses that humans would prefer).
```

In [ ]:
# Simulate the RLHF preference collection process

import numpy as np

# Simulated preference data
preference_examples = [
    {
        "prompt": "What's the weather like today?",
        "response_a": "I don't have access to real-time weather data. You can check a weather app or website for current conditions in your area.",
        "response_b": "The weather is sunny and 72 degrees.",
        "preferred": "A",
        "reason": "A is honest about limitations; B makes up information"
    },
    {
        "prompt": "Write a short poem about cats.",
        "response_a": "Cats are animals that have four legs and whiskers.",
        "response_b": "Soft paws on silent floor,\nA gentle purr, a playful roar,\nEyes like moons in velvet night,\nA cat curls up, the world feels right.",
        "preferred": "B",
        "reason": "B is creative and actually writes a poem; A just states facts"
    },
    {
        "prompt": "How do I pick a lock?",
        "response_a": "Here's a step-by-step guide to pick a lock: First, get a tension wrench...",
        "response_b": "I can't provide instructions for picking locks as it could be used for illegal purposes. If you're locked out, I'd recommend calling a licensed locksmith.",
        "preferred": "B",
        "reason": "B is safe and responsible; A could enable harmful behavior"
    },
]

print("RLHF Preference Collection Examples")
print("=" * 60)
print("(Humans compare pairs of responses and pick the better one)")

sep = "─" * 60

for i, ex in enumerate(preference_examples, 1):
    print(f"\n{sep}")
    print(f"Example {i}")
    print(f"  Prompt: \"{ex['prompt']}\"")
    print(f"\n  Response A: \"{ex['response_a'][:80]}...\"" 
          if len(ex['response_a']) > 80 else f"\n  Response A: \"{ex['response_a']}\"")
    print(f"  Response B: \"{ex['response_b'][:80]}...\"" 
          if len(ex['response_b']) > 80 else f"  Response B: \"{ex['response_b']}\"")
    print(f"\n  Human preference: Response {ex['preferred']}")
    print(f"  Reason: {ex['reason']}")

print(f"\n{'=' * 60}")
print("The reward model learns from thousands of these comparisons.")
print("It learns that good responses are: helpful, honest, and safe.")

In [ ]:
# Visualize the effect of RLHF on response quality

import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# -- 1. Response quality improvement --
ax = axes[0]

stages = ['Base\nModel', 'After SFT', 'After RLHF']
metrics = {
    'Helpfulness': [30, 65, 90],
    'Honesty': [40, 60, 85],
    'Safety': [20, 55, 88],
}
colors = ['#4CAF50', '#2196F3', '#F44336']
x = np.arange(len(stages))
width = 0.25

for i, (metric, scores) in enumerate(metrics.items()):
    bars = ax.bar(x + i*width, scores, width, label=metric, color=colors[i],
                  edgecolor='black', linewidth=1)
    for bar, score in zip(bars, scores):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{score}%', ha='center', fontsize=8, fontweight='bold')

ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('Model Quality at Each Stage', fontweight='bold', fontsize=14)
ax.set_xticks(x + width)
ax.set_xticklabels(stages, fontsize=11)
ax.legend(fontsize=10)
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.3, axis='y')

# -- 2. What RLHF optimizes for --
ax = axes[1]

categories = ['Helpful', 'Honest', 'Harmless', 'Concise', 'Engaging', 'Accurate']
before_rlhf = [5, 4, 3, 4, 5, 6]
after_rlhf = [9, 8, 9, 8, 8, 8]

y = np.arange(len(categories))
height = 0.35

bars1 = ax.barh(y - height/2, before_rlhf, height, label='Before RLHF',
                color='#FFCDD2', edgecolor='#C62828', linewidth=1)
bars2 = ax.barh(y + height/2, after_rlhf, height, label='After RLHF',
                color='#C8E6C9', edgecolor='#2E7D32', linewidth=1)

ax.set_xlabel('Score (out of 10)', fontsize=12)
ax.set_title('What RLHF Improves', fontweight='bold', fontsize=14)
ax.set_yticks(y)
ax.set_yticklabels(categories, fontsize=11)
ax.legend(fontsize=10)
ax.set_xlim(0, 11)
ax.grid(True, alpha=0.3, axis='x')

plt.suptitle('The Impact of Instruction Tuning + RLHF',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("RLHF dramatically improves the model's helpfulness, honesty, and safety.")
print("It is the difference between a raw model and something like ChatGPT.")

---
## 6. DPO: A Simpler Alternative to RLHF

RLHF is effective but complex — it requires training a separate reward model and using reinforcement learning (which can be unstable).

In 2023, researchers proposed **DPO (Direct Preference Optimization)** — a simpler way to achieve similar results.

### RLHF vs DPO

```
  RLHF (Complex — 3 models needed!):
  ┌───────────┐    ┌───────────┐    ┌───────────┐
  │  Step 1:  │    │  Step 2:  │    │  Step 3:  │
  │  Collect   │───>│  Train    │───>│  Use RL   │
  │ preferences│    │  reward   │    │  to train │
  │           │    │  model    │    │  the LLM  │
  └───────────┘    └───────────┘    └───────────┘
  
  Needs: LLM + Reward Model + RL algorithm
  Complexity: HIGH
  
  
  DPO (Simple — 1 model needed!):
  ┌───────────┐    ┌───────────────────────────────┐
  │  Step 1:  │    │  Step 2:                       │
  │  Collect   │───>│  Directly train the LLM on     │
  │ preferences│    │  preferred vs. rejected answers│
  └───────────┘    └───────────────────────────────┘
  
  Needs: Just the LLM!
  Complexity: LOW (it is just supervised learning!)
```

### How DPO Works

Instead of training a separate reward model, DPO directly teaches the LLM:

```
  For each preference pair:
  
  Prompt:    "Explain gravity simply"
  Preferred: "It is the force that pulls things down..."  ← Make MORE likely
  Rejected:  "According to general relativity..."        ← Make LESS likely
  
  The loss function directly:
  • Increases probability of preferred responses
  • Decreases probability of rejected responses
  
  No reward model needed! No RL needed! Just standard training!
```

### DPO vs RLHF Comparison

| Feature | RLHF | DPO |
|---------|------|-----|
| Quality | Excellent | Comparable |
| Complexity | High (3 models) | Low (1 model) |
| Stability | Can be unstable | Stable |
| Memory | Very high | Moderate |
| Ease of implementation | Difficult | Easy |
| Used by | OpenAI (ChatGPT) | Many open-source models |

---
## 7. Creating Your Own Instruction Dataset

Want to instruction-tune a model for your own use case? Here is how to create a good dataset:

### Step-by-Step Guide

```
  1. DEFINE YOUR TASK
     What should the model do?
     Examples:
     • Customer support bot → Questions about your product
     • Medical assistant → Medical terminology explanations
     • Code reviewer → Find bugs in code
  
  2. WRITE DIVERSE EXAMPLES
     Aim for 100-10,000 high-quality examples
     Cover different:
     • Difficulty levels (easy, medium, hard)
     • Topics within your domain
     • Formats (questions, commands, conversations)
  
  3. ENSURE QUALITY
     • Have experts review responses
     • Remove duplicates
     • Fix grammar and formatting
     • Include edge cases
  
  4. FORMAT CONSISTENTLY
     Use a standard format (Alpaca, ChatML, etc.)
     Keep formatting consistent across all examples
```

In [ ]:
# Let's create a small instruction dataset for a custom use case
# Example: A customer support bot for a fictional coffee shop

import json

coffee_shop_dataset = [
    {
        "instruction": "What are your opening hours?",
        "input": "",
        "output": "We're open Monday through Friday from 7 AM to 7 PM, "
                  "and Saturday-Sunday from 8 AM to 5 PM. Holiday hours may vary."
    },
    {
        "instruction": "What's the difference between a latte and a cappuccino?",
        "input": "",
        "output": "Both are espresso-based drinks with steamed milk, but they differ in milk ratio:\n"
                  "- Latte: More steamed milk, thin layer of foam (creamy and smooth)\n"
                  "- Cappuccino: Equal parts espresso, steamed milk, and foam (stronger, more textured)"
    },
    {
        "instruction": "I have a complaint about my order.",
        "input": "I ordered a caramel macchiato but received a regular latte.",
        "output": "I'm sorry about the mix-up with your order! That's not the experience "
                  "we want you to have. Please bring the drink back to any of our locations "
                  "and we'll make you a fresh caramel macchiato right away, on the house. "
                  "We apologize for the inconvenience."
    },
    {
        "instruction": "Do you have dairy-free options?",
        "input": "",
        "output": "Yes! We offer several dairy-free milk alternatives at no extra charge:\n"
                  "- Oat milk (our most popular alternative)\n"
                  "- Almond milk\n"
                  "- Soy milk\n"
                  "- Coconut milk\n"
                  "Any of our drinks can be made with these alternatives."
    },
    {
        "instruction": "Can I place a large order for an office meeting?",
        "input": "We need coffee for 20 people next Tuesday morning.",
        "output": "We'd love to help with your office meeting! For groups of 20, I'd recommend:\n"
                  "- Our Carafe Service: Each carafe serves 8-10 cups ($25 each)\n"
                  "- You'd need about 2-3 carafes\n"
                  "- We can include assorted pastries too\n\n"
                  "Please place catering orders at least 24 hours in advance. "
                  "Call us at (555) 123-4567 or order through our app!"
    },
]

print("Custom Instruction Dataset: Coffee Shop Support Bot")
print("=" * 60)
print(f"Total examples: {len(coffee_shop_dataset)}")
print(f"(In practice, you would want 100-10,000+ examples)\n")

for i, ex in enumerate(coffee_shop_dataset, 1):
    print(f"Example {i}:")
    print(f"  Customer: \"{ex['instruction']}\"")
    if ex['input']:
        print(f"  Context:  \"{ex['input']}\"")
    response_preview = ex['output'][:100]
    print(f"  Bot:      \"{response_preview}...\"" if len(ex['output']) > 100 else f"  Bot:      \"{ex['output']}\"")
    print()

print("This dataset would be saved as JSON and used for fine-tuning!")
print("You could fine-tune a model like LLaMA with LoRA/QLoRA using this data.")

---
## 8. Putting It All Together: The Complete Fine-Tuning Landscape

Now you have learned all the major concepts! Here is how everything fits together:

```
  THE COMPLETE FINE-TUNING LANDSCAPE
  ═════════════════════════════════
  
  1. Pre-trained Model (start here)
     │
     ├── Do you need to customize it?
     │   ├── NO → Use it as-is with prompt engineering
     │   └── YES → Continue...
     │
     ├── HOW to fine-tune? (the method)
     │   ├── Full Fine-Tuning → Best quality, most expensive
     │   ├── LoRA → Great quality, affordable
     │   └── QLoRA → Great quality, cheapest (most popular!)
     │
     ├── WHAT data to use? (the dataset)
     │   ├── Task-specific data → Classification, generation, etc.
     │   └── Instruction data → Make it follow instructions
     │
     └── HOW to align with humans? (optional but powerful)
         ├── RLHF → Complex but proven (ChatGPT)
         └── DPO → Simple and effective (most open-source models)
```

---
## Key Takeaways

1. **Raw language models just predict the next word** — they are not naturally helpful

2. **Instruction tuning (SFT)** teaches models to follow instructions using (instruction, response) pairs

3. **RLHF** further improves the model by learning from human preferences (which response is better?)

4. **DPO** is a simpler alternative to RLHF that achieves similar results without needing a reward model

5. **The ChatGPT recipe**: Pre-training + SFT + RLHF

6. **You can create your own instruction datasets** for custom use cases (customer support, medical, legal, etc.)

7. **Dataset quality matters more than size** — 1,000 great examples beat 100,000 poor ones

## Congratulations!

You have completed the Fine-Tuning module! Here is a summary of what you learned:

| Notebook | Key Concept |
|----------|------------|
| 01 - What is Fine-Tuning? | Transfer learning, pre-trained models, types of fine-tuning |
| 02 - Full Fine-Tuning | Gradient descent, updating all parameters, costs |
| 03 - LoRA | Low-rank adapters, parameter-efficient fine-tuning |
| 04 - QLoRA | Quantization + LoRA, fine-tuning on consumer GPUs |
| 05 - Instruction Tuning | SFT, RLHF, DPO, how ChatGPT was made |

For interview-depth coverage of SFT loss masking, RLHF objective, and DPO derivation, read [instruction-tuning-interview.md](./instruction-tuning-interview.md).

---

[Back to Module README](./README.md)

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)